# Simple Model Experiments

Goal: start dead simple, add complexity only when it helps.

**Models in order of complexity:**
1. Logistic Regression (aggregated features, no sequence)
2. Small MLP (aggregated features)
3. Tiny Transformer (1 layer, d=32) with time-based positional encoding
4. Compare all three against the full model

**Key hypothesis:** 1232 training samples is too few for a 163k-param transformer.
A ~10k-param model should generalise better.

In [ ]:
import pickle
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss
import matplotlib.pyplot as plt
import sys, os
sys.path.insert(0, os.path.abspath('..'))

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

## 1. Load Data

In [ ]:
def load(path):
    with open(path, 'rb') as f:
        return pickle.load(f)

train = load('../data/processed/train_samples.pkl')
val   = load('../data/processed/val_samples.pkl')
test  = load('../data/processed/test_samples.pkl')

print(f'Train: {len(train)}  Val: {len(val)}  Test: {len(test)}')
print()

# Inspect one sample
s = train[0]
print('Sample keys:     ', list(s.keys()))
print('History A length:', len(s['history_a']))
print('Feature keys:    ', list(s['history_a'][0].keys()) if s['history_a'] else 'empty')
print('Scalars shape:   ', s['history_a'][0]['scalars'].shape if s['history_a'] else 'n/a')
print('Meta IDs in hist:', [e['meta_id'] for e in s['history_a'][:5]], '...')

## 2. Feature Engineering â€” Aggregate the 20-match history

For simple models (logistic regression, MLP), we don't feed a sequence.
Instead we summarise each team's history into a fixed-size vector:
- **Mean** of each scalar feature over non-padded matches
- **Recency-weighted mean** (recent matches count more)
- **Last-match features** (most recent form snapshot)

Final feature vector per team: 11 (mean) + 11 (recency-weighted) + 11 (last) = 33 dims
Concatenated for both teams: 66 dims total

In [ ]:
FEATURE_NAMES = [
    'win_binary', 'maps_won', 'maps_lost', 'maps_played',
    'map_win_rate', 'map_score_diff', 'tournament_tier',
    'bracket_stage', 'opponent_elo', 'days_since', 'max_round_streak'
]
N_SCALARS = 11


def aggregate_history(history: list) -> np.ndarray:
    """Turn up-to-20 match dicts into a fixed 33-dim vector."""
    real = [e for e in history if e['scalars'].sum() != 0]  # exclude padded
    if not real:
        return np.zeros(N_SCALARS * 3, dtype=np.float32)

    mat = np.stack([e['scalars'] for e in real])   # (n, 11)

    # Plain mean
    mean_feats = mat.mean(axis=0)

    # Recency-weighted mean â€” exponential decay, most recent = highest weight
    n = len(mat)
    weights = np.exp(np.linspace(-2, 0, n))        # older â†’ lower weight
    weights /= weights.sum()
    recency_feats = (mat * weights[:, None]).sum(axis=0)

    # Last match snapshot
    last_feats = mat[-1]

    return np.concatenate([mean_feats, recency_feats, last_feats]).astype(np.float32)


def build_xy(samples):
    X, y = [], []
    for s in samples:
        fa = aggregate_history(s['history_a'])
        fb = aggregate_history(s['history_b'])
        # Symmetric: use difference + sum so order doesn't matter
        feat = np.concatenate([fa - fb, fa + fb])
        X.append(feat)
        y.append(1 if s['winner'] == 0 else 0)   # 1 = team_a wins
    return np.stack(X), np.array(y)


X_train, y_train = build_xy(train)
X_val,   y_val   = build_xy(val)
X_test,  y_test  = build_xy(test)

print('X_train shape:', X_train.shape)  # (1232, 66)
print('Class balance (train):', y_train.mean().round(3))

## 3. Model 1 â€” Logistic Regression

In [ ]:
def evaluate(probs, labels, name):
    preds = (probs >= 0.5).astype(int)
    acc   = (preds == labels).mean()
    auc   = roc_auc_score(labels, probs)
    brier = brier_score_loss(labels, probs)
    ll    = log_loss(labels, probs)
    print(f'{name:30s}  acc={acc:.3f}  auc={auc:.3f}  brier={brier:.3f}  logloss={ll:.3f}')
    return {'acc': acc, 'auc': auc, 'brier': brier, 'logloss': ll}


lr = LogisticRegression(C=0.1, max_iter=1000, random_state=42)
lr.fit(X_train, y_train)

val_probs  = lr.predict_proba(X_val)[:, 1]
test_probs = lr.predict_proba(X_test)[:, 1]

lr_val  = evaluate(val_probs,  y_val,  'LogReg  (val)')
lr_test = evaluate(test_probs, y_test, 'LogReg  (test)')

## 4. Model 2 â€” Small MLP

In [ ]:
class SmallMLP(nn.Module):
    def __init__(self, in_dim=66):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 32),     nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.net(x)

print(f'MLP params: {sum(p.numel() for p in SmallMLP().parameters()):,}')

In [ ]:
class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]


def train_mlp(model, X_tr, y_tr, X_val, y_val, epochs=300, lr=1e-3, patience=30):
    model = model.to(DEVICE)
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit  = nn.BCEWithLogitsLoss()
    loader = DataLoader(TabularDataset(X_tr, y_tr), batch_size=32, shuffle=True)

    best_brier, best_state, wait = 1.0, None, 0
    train_losses, val_briers = [], []

    Xv = torch.tensor(X_val, dtype=torch.float32).to(DEVICE)
    yv = torch.tensor(y_val, dtype=torch.float32).to(DEVICE)

    for ep in range(1, epochs + 1):
        model.train()
        total = 0
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(xb).squeeze(), yb)
            loss.backward()
            opt.step()
            total += loss.item() * len(yb)
        sched.step()
        train_losses.append(total / len(y_tr))

        model.eval()
        with torch.no_grad():
            probs = torch.sigmoid(model(Xv).squeeze()).cpu().numpy()
        brier = brier_score_loss(y_val, probs)
        val_briers.append(brier)

        if brier < best_brier - 1e-5:
            best_brier, best_state, wait = brier, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            wait += 1
            if wait >= patience:
                print(f'  Early stop at epoch {ep}')
                break

    model.load_state_dict(best_state)
    return model, train_losses, val_briers


mlp = SmallMLP(in_dim=X_train.shape[1])
mlp, mlp_train_losses, mlp_val_briers = train_mlp(mlp, X_train, y_train, X_val, y_val)

mlp.eval()
with torch.no_grad():
    mlp_val_probs  = torch.sigmoid(mlp(torch.tensor(X_val,  dtype=torch.float32).to(DEVICE)).squeeze()).cpu().numpy()
    mlp_test_probs = torch.sigmoid(mlp(torch.tensor(X_test, dtype=torch.float32).to(DEVICE)).squeeze()).cpu().numpy()

mlp_val  = evaluate(mlp_val_probs,  y_val,  'MLP     (val)')
mlp_test = evaluate(mlp_test_probs, y_test, 'MLP     (test)')

## 5. Model 3 â€” Tiny Transformer

Key design changes vs full model:
- **d_model = 32** (was 64)
- **1 encoder layer** (was 3)
- **2 attention heads** (was 4)
- **dim_feedforward = 64** (was 256)
- **No learned positional embedding** â€” use actual `days_since_match` (feature index 9) as a continuous time signal so the model knows how recent each match is
- ~8k parameters total (was 163k)

In [ ]:
class TinyMatchEncoder(nn.Module):
    """Projects one match token to d_model dims.
    Uses days_since (feature 9) as a continuous time embedding instead of
    a learned slot embedding — this respects actual recency.
    """
    def __init__(self, num_scalars=11, num_maps=12, map_embed_dim=8,
                 num_metas=37, d_model=32):
        super().__init__()
        self.map_embed  = nn.Embedding(num_maps, map_embed_dim)
        self.meta_embed = nn.Embedding(num_metas, d_model)
        # Project scalars + map embedding to d_model
        self.proj = nn.Sequential(
            nn.Linear(num_scalars + map_embed_dim, d_model),
            nn.LayerNorm(d_model)
        )
        # Continuous time MLP: days_since -> d_model
        # Encodes actual recency, not just slot position
        self.time_proj = nn.Sequential(
            nn.Linear(1, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model)
        )

    def forward(self, scalars, map_idx, meta_idx):
        # scalars: (B, 20, 11)
        map_emb = self.map_embed(map_idx)               # (B, 20, 8)
        x = torch.cat([scalars, map_emb], dim=-1)       # (B, 20, 19)
        x = self.proj(x)                                # (B, 20, 32)

        # Time embedding from actual days_since (feature index 9)
        days = scalars[:, :, 9:10]                      # (B, 20, 1)
        x = x + self.time_proj(days)                    # (B, 20, 32)

        # Meta period embedding
        x = x + self.meta_embed(meta_idx)               # (B, 20, 32)
        return x


class TinyTransformer(nn.Module):
    def __init__(self, d_model=32, nhead=2, dim_feedforward=64,
                 num_layers=1, dropout=0.1, num_metas=37):
        super().__init__()
        self.encoder_input = TinyMatchEncoder(
            d_model=d_model, map_embed_dim=8, num_metas=num_metas
        )
        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout, activation='gelu',
            batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(
            layer, num_layers=num_layers, enable_nested_tensor=False
        )
        # Classifier: symmetric [a-b, a+b] -> logit
        self.head = nn.Linear(d_model * 2, 1)

    def encode_team(self, scalars, map_idx, meta_idx, pad_mask):
        x = self.encoder_input(scalars, map_idx, meta_idx)   # (B, 20, d)

        # Guard: if ALL positions are masked for a sample, PyTorch softmax
        # produces NaN (softmax of all -inf). Unmask at least one position
        # per sample so attention can attend somewhere.
        all_masked = pad_mask.all(dim=1, keepdim=True)       # (B, 1) bool
        safe_mask  = pad_mask & ~all_masked.expand_as(pad_mask)

        x = self.transformer(x, src_key_padding_mask=safe_mask)

        # Mean pool over truly non-padded positions only
        real = (~pad_mask).float().unsqueeze(-1)              # (B, 20, 1)
        return (x * real).sum(1) / real.sum(1).clamp(min=1)  # (B, d)

    def forward(self, scalars_a, map_idx_a, pad_mask_a, meta_idx_a,
                      scalars_b, map_idx_b, pad_mask_b, meta_idx_b):
        ra = self.encode_team(scalars_a, map_idx_a, meta_idx_a, pad_mask_a)
        rb = self.encode_team(scalars_b, map_idx_b, meta_idx_b, pad_mask_b)
        return self.head(torch.cat([ra - rb, ra + rb], dim=-1))


tiny = TinyTransformer()
print(f'TinyTransformer params: {sum(p.numel() for p in tiny.parameters()):,}')

In [ ]:
from src.data.dataset import MatchDataset
from src.data.augmentation import TeamSwapDataset

train_ds = TeamSwapDataset(MatchDataset(train))
val_ds   = MatchDataset(val)
test_ds  = MatchDataset(test)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=0)

print(f'Train batches: {len(train_loader)}  (augmented: {len(train_ds)} samples)')

In [ ]:
def train_transformer(model, train_loader, val_loader, epochs=200, lr=1e-3, patience=30):
    model = model.to(DEVICE)
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-5)
    # BCEWithLogitsLoss doesn't support label_smoothing â€” apply it manually.
    # Smooth target: y_smooth = y * (1 - eps) + eps/2   (eps = 0.1)
    crit  = nn.BCEWithLogitsLoss()
    EPS   = 0.1

    best_brier, best_state, wait = 1.0, None, 0
    train_losses, val_briers = [], []

    for ep in range(1, epochs + 1):
        # --- Train ---
        model.train()
        total = 0
        for batch in train_loader:
            b = {k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
            opt.zero_grad()
            logits = model(
                b['scalars_a'], b['map_idx_a'], b['pad_mask_a'], b['meta_idx_a'],
                b['scalars_b'], b['map_idx_b'], b['pad_mask_b'], b['meta_idx_b'],
            ).squeeze(-1)
            smooth_labels = b['label'] * (1 - EPS) + EPS / 2
            loss = crit(logits, smooth_labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += loss.item() * len(b['label'])
        sched.step()
        train_losses.append(total / len(train_loader.dataset))

        # --- Val ---
        model.eval()
        probs, labels = [], []
        with torch.no_grad():
            for batch in val_loader:
                b = {k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
                logits = model(
                    b['scalars_a'], b['map_idx_a'], b['pad_mask_a'], b['meta_idx_a'],
                    b['scalars_b'], b['map_idx_b'], b['pad_mask_b'], b['meta_idx_b'],
                ).squeeze(-1)
                probs.append(torch.sigmoid(logits).cpu().numpy())
                labels.append(b['label'].cpu().numpy())
        probs  = np.concatenate(probs)
        labels = np.concatenate(labels)
        brier  = brier_score_loss(labels, probs)
        val_briers.append(brier)

        if brier < best_brier - 1e-5:
            best_brier = brier
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(f'  Early stop at epoch {ep}  (best brier={best_brier:.4f})')
                break

        if ep % 20 == 0:
            print(f'  Epoch {ep:3d} | train_loss={train_losses[-1]:.4f} | val_brier={brier:.4f}')

    model.load_state_dict(best_state)
    return model, train_losses, val_briers


def eval_transformer(model, loader):
    model.eval()
    probs, labels = [], []
    with torch.no_grad():
        for batch in loader:
            b = {k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
            # Symmetric: average both orderings
            l_ab = model(
                b['scalars_a'], b['map_idx_a'], b['pad_mask_a'], b['meta_idx_a'],
                b['scalars_b'], b['map_idx_b'], b['pad_mask_b'], b['meta_idx_b'],
            ).squeeze(-1)
            l_ba = model(
                b['scalars_b'], b['map_idx_b'], b['pad_mask_b'], b['meta_idx_b'],
                b['scalars_a'], b['map_idx_a'], b['pad_mask_a'], b['meta_idx_a'],
            ).squeeze(-1)
            p = (torch.sigmoid(l_ab) + (1 - torch.sigmoid(l_ba))) / 2
            probs.append(p.cpu().numpy())
            labels.append(b['label'].cpu().numpy())
    return np.concatenate(probs), np.concatenate(labels)


print('Training TinyTransformer...')
tiny, tiny_train_losses, tiny_val_briers = train_transformer(tiny, train_loader, val_loader)

In [ ]:
tiny_val_probs,  tiny_val_labels  = eval_transformer(tiny, val_loader)
tiny_test_probs, tiny_test_labels = eval_transformer(tiny, test_loader)

tiny_val  = evaluate(tiny_val_probs,  tiny_val_labels,  'TinyTransformer (val)')
tiny_test = evaluate(tiny_test_probs, tiny_test_labels, 'TinyTransformer (test)')

## 6. Summary â€” All Models vs Targets

In [ ]:
print('=' * 75)
print(f'{"Model":<30} {"Acc":>6} {"AUC":>6} {"Brier":>7} {"LogLoss":>9}')
print('-' * 75)

results = [
    ('LogReg         (val)',  lr_val),
    ('LogReg         (test)', lr_test),
    ('MLP            (val)',  mlp_val),
    ('MLP            (test)', mlp_test),
    ('TinyTransformer(val)',  tiny_val),
    ('TinyTransformer(test)', tiny_test),
]

for name, m in results:
    print(f'{name:<30} {m["acc"]:>6.3f} {m["auc"]:>6.3f} {m["brier"]:>7.3f} {m["logloss"]:>9.3f}')

print('=' * 75)
print(f'{"TARGETS":<30} {"0.60":>6} {"0.65":>6} {"0.220":>7} {"0.650":>9}')

In [ ]:
# Training curves â€” TinyTransformer
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(tiny_train_losses, label='train loss')
axes[0].set_title('TinyTransformer â€” Train Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(tiny_val_briers, label='val brier', color='orange')
axes[1].axhline(0.22, color='green', linestyle='--', label='target (<0.22)')
axes[1].axhline(0.25, color='red',   linestyle='--', label='random (0.25)')
axes[1].set_title('TinyTransformer â€” Val Brier Score')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Feature Importance (Logistic Regression)

Since logistic regression is interpretable, we can see which features the model weights most heavily.

In [ ]:
# Coefficients correspond to [mean_diff, recency_diff, last_diff, mean_sum, recency_sum, last_sum]
# We only look at the first 33 (diff features) â€” positive = helps team A
feat_labels = (
    [f'mean_{n}' for n in FEATURE_NAMES] +
    [f'recency_{n}' for n in FEATURE_NAMES] +
    [f'last_{n}' for n in FEATURE_NAMES] +
    [f'sum_mean_{n}' for n in FEATURE_NAMES] +
    [f'sum_rec_{n}'  for n in FEATURE_NAMES] +
    [f'sum_last_{n}' for n in FEATURE_NAMES]
)

coefs = lr.coef_[0]
# Top 15 most impactful features
top_idx = np.argsort(np.abs(coefs))[-15:][::-1]

plt.figure(figsize=(10, 5))
bars = plt.barh([feat_labels[i] for i in top_idx], coefs[top_idx],
                color=['steelblue' if c > 0 else 'tomato' for c in coefs[top_idx]])
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Top 15 Feature Weights (Logistic Regression)')
plt.xlabel('Coefficient (positive = helps team A win)')
plt.tight_layout()
plt.show()